# Modelling (`test_shop` + `set_a` + `set_c` + Expanded Model Suite): Rolling Backtest & Submission

Paired with `preprocessing_test_shop_set_a_c.ipynb`, this notebook trains global models on **`test_shop` + `set_a` + `set_c`** (with `test_shop` weighted `1.0` and `set_a`/`set_c` weighted `0.35`), evaluates exclusively on **`test_shop`** (`features_wide.parquet` + `history.parquet`), and exports the winning `w1..w13` submission.

**Models & Ensembles Included**

| Model | Family | Loss / Rule | Role |
|---|---|---|---|
| `median_13w` | Reference | 13-week rolling median | Main baseline to beat |
| `naive_last`, `seasonal_naive_52` | Reference | Last week / same 13 weeks last year | Calibration baselines |
| `lightgbm` | Gradient-boosted trees (L1) | L1 loss on scaled target (`num_leaves=63`, `min_data_in_leaf=500`) | Primary tree learner |
| `lightgbm_tweedie` | Gradient-boosted trees (Tweedie) | Tweedie variance power `1.3` (`log` link) | Zero-inflated / skewed count specialist |
| `catboost` | Oblivious gradient-boosted trees (MAE) | Symmetric trees with L1 (`MAE`) loss | Strong regularization & low variance on tabular features |
| `hist_gbdt` | Histogram GBDT (`scikit-learn`) | `loss='absolute_error'` (L1) | Fast native L1 tree diversity |
| `linear_l1` | Linear (`SGDRegressor`) | L1 (`epsilon=0.0`) on `signed_log` scaled features | Linear benchmark |
| `mlp` | Neural network (`MLPRegressor`) | Squared error on `log1p(target)` | Non-linear neural diversity |
| `tsb` | Intermittent-demand smoothing | Teunter–Syntetos–Babai median rule | Intermittent & lumpy item specialist |
| `croston_sba` | Intermittent-demand smoothing | Syntetos–Boylan Approximation (`(1 - alpha/2) * z / p`) | De-biased Croston intermittent baseline |
| `ens_mean(lgb+cat)` | Ensemble | Mean of `lightgbm` + `catboost` | Tree-family blend |
| `ens_median(top_diverse)` | Ensemble | Median across `lightgbm`, `lightgbm_tweedie`, `catboost`, `hist_gbdt`, `linear_l1`, `tsb` | Cross-family robust blend |
| `class_routed` | Demand-class router | Uses `tsb` on `intermittent`/`lumpy` articles and `ens_mean(lgb+cat)` (or `lightgbm`) on `smooth`/`erratic` articles | Exploits per-class strengths (§11) |

In [ ]:
%pip install -q numpy pandas matplotlib pyarrow scikit-learn lightgbm catboost

In [ ]:
import gc
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

IN_COLAB = Path('/content').exists()
SUBMISSION_DIR = Path('/content/submissions') if IN_COLAB else Path('submissions')

H = 13
SID = 'series_id'
SEED = 0

# Plot style
BLUE, GREY, INK = '#2a78d6', '#b5b4ae', '#52514e'
plt.rcParams.update({
    'figure.figsize': (11, 4), 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': GREY, 'axes.labelcolor': INK,
    'xtick.color': INK, 'ytick.color': INK,
    'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
})

## 1. Load `features_wide.parquet` (`test_shop` + `set_a` + `set_c`) and `history.parquet`

- Automatically locates `features_wide.parquet` (generated by `preprocessing_test_shop_set_a_c.ipynb`) and `history.parquet` (`test_shop`) in `/content/`, `/content/processed/`, `./processed/`, or `.`.
- Keeps `test_shop`, `set_a`, and `set_c` for global training while building the `Y` target matrix strictly from `test_shop`.

In [ ]:
USE_UPLOAD_DIALOG = False   # Set True only if you want the inline Choose Files button instead of the left sidebar


def locate_parquet(stem, explicit_path=None):
    if explicit_path is not None:
        p = Path(explicit_path)
        if p.exists() and p.stat().st_size > 0:
            return p
        raise FileNotFoundError(f'Explicit path not ready: {p}')

    roots = [Path('/content'), Path('.')] if IN_COLAB else [Path('.'), Path('data'), Path('data/hackathon-retail-sets')]
    matches = []
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob('*.parquet'):
            if p.name.startswith(stem):
                matches.append(p)

    valid = [p for p in matches if p.stat().st_size > 0]
    if valid:
        valid.sort(key=lambda p: (p.name != f'{stem}.parquet', -p.stat().st_mtime))
        return valid[0]

    if matches:
        raise RuntimeError(
            f'Found `{matches[0]}` but its size is 0 bytes — wait for the Colab upload progress circle to finish, then re-run.'
        )

    if IN_COLAB and USE_UPLOAD_DIALOG:
        from google.colab import files
        print(f'Select `{stem}.parquet` in the file picker below:')
        uploaded = files.upload()
        for k in uploaded:
            if k.endswith('.parquet'):
                return Path('/content') / k

    found_any = [f'{p} ({p.stat().st_size / 1e6:.1f} MB)' for r in roots if r.exists() for p in r.rglob('*.parquet')]
    raise FileNotFoundError(
        f'Could not find `{stem}.parquet`. Upload `features_wide.parquet` and `history.parquet` in Colab Files sidebar.\n'
        f'Parquet files currently visible: {found_any or "none"}'
    )


def parse_week_start(s):
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_datetime(s, unit='D', origin='1970-01-01')
    return pd.to_datetime(s)


t0 = time.perf_counter()
FEATURES_PATH = locate_parquet('features_wide')
HISTORY_PATH = locate_parquet('history')
print(f'Loading features: {FEATURES_PATH} ({FEATURES_PATH.stat().st_size / 1e6:.1f} MB)')
print(f'Loading history:  {HISTORY_PATH} ({HISTORY_PATH.stat().st_size / 1e6:.1f} MB)')

frame = pd.read_parquet(FEATURES_PATH, engine='pyarrow')
frame['week_start'] = parse_week_start(frame['week_start'])
allowed_sources = {'test_shop', 'set_a', 'set_c'}
if 'source' in frame.columns:
    frame = frame[frame['source'].isin(allowed_sources)].reset_index(drop=True)
    frame['source'] = pd.Categorical(frame['source'], categories=['test_shop', 'set_a', 'set_c'])
else:
    frame['source'] = pd.Categorical(['test_shop'] * len(frame), categories=['test_shop', 'set_a', 'set_c'])

print(f'Loaded combined feature frame in {time.perf_counter() - t0:.1f}s: '
      f'{len(frame):,} rows x {frame.shape[1]} cols | {frame[SID].nunique():,} total series')
display(frame.groupby('source', observed=True).agg(
    rows=('units', 'size'),
    series=(SID, 'nunique'),
    min_week=('week_start', 'min'),
    max_week=('week_start', 'max'),
    sample_weight=('sample_weight', 'first'),
))

ts = pd.read_parquet(HISTORY_PATH, engine='pyarrow')
ts['week_start'] = parse_week_start(ts['week_start']).dt.to_period('W-SUN').dt.start_time
ts['article_id'] = ts['article_id'].astype(str)
Y = ts.pivot(index='article_id', columns='week_start', values='units').sort_index(axis=1)
print(f'Test shop history matrix: {Y.shape[0]:,} articles x {Y.shape[1]} weeks ({Y.columns[0].date()} -> {Y.columns[-1].date()})')

## 2. Fast helpers (vectorized `to_long` + MASE metric)

In [ ]:
META_COLS = ['source', SID, 'week_start', 'units', 'censored', 'store_id', 'article_id', 'scale', 'is_outlier',
             'sample_weight', 'supplier_id', 'product_group', 'division', 'department', 'department_group',
             'store_format', 'category', 'subcategory', 'segment']
LONG_ID_COLS = ['source', SID, 'article_id', 'week_start', 'scale', 'sample_weight']


def rolling_cutoffs(frame, n_windows=4, step=H, horizon=H, source='test_shop'):
    last = frame.loc[frame['source'] == source, 'week_start'].max()
    return [last - pd.Timedelta(weeks=horizon + i * step) for i in range(n_windows)]


def split_frame(frame, cutoff=None, horizon=H, valid_source='test_shop'):
    is_v = frame['source'] == valid_source
    if cutoff is None:
        cutoff = frame.loc[is_v, 'week_start'].max()
    target_end = frame['week_start'] + pd.Timedelta(weeks=horizon)
    # Keep all auxiliary (set_a, set_c) rows whose targets don't leak past cutoff, plus non-leaking test_shop rows
    train = frame[target_end <= cutoff]
    valid = frame[is_v & (frame['week_start'] == cutoff)]
    return train.reset_index(drop=True), valid.reset_index(drop=True)


def subsample_origins(frame, every=None, min_history_weeks=1):
    keep = pd.Series(True, index=frame.index)
    if every:
        k = frame['source'].astype(str).map(every).fillna(1).astype(int)
        from_end = frame.groupby(SID, sort=False, observed=True).cumcount(ascending=False)
        keep &= (from_end % k == 0)
    if 'age_weeks' in frame:
        keep &= frame['age_weeks'] >= min_history_weeks - 1
    return frame[keep].reset_index(drop=True)


def to_long(frame, horizon=H, drop_missing_target=True, id_cols=LONG_ID_COLS):
    """Vectorized wide-to-long expansion across horizons 1..horizon."""
    n = len(frame)
    feats = [c for c in frame.columns if c not in META_COLS and not c.startswith(('target_h', 'seasonal_h'))]
    base_cols = [c for c in id_cols if c in frame.columns] + feats

    rep_idx = np.tile(np.arange(n, dtype=np.int64), horizon)
    long = frame[base_cols].iloc[rep_idx].reset_index(drop=True)
    long['h'] = np.repeat(np.arange(1, horizon + 1, dtype=np.int8), n)

    if all(f'seasonal_h{h}' in frame.columns for h in range(1, horizon + 1)):
        s_cols = [f'seasonal_h{h}' for h in range(1, horizon + 1)]
        long['seasonal_lag52'] = frame[s_cols].to_numpy().T.ravel()

    if any(f'target_h{h}' in frame.columns for h in range(1, horizon + 1)):
        t_cols = [f'target_h{h}' for h in range(1, horizon + 1)]
        long['target'] = frame[t_cols].to_numpy().T.ravel()

    long[[SID, 'article_id']] = long[[SID, 'article_id']].astype('category')
    if drop_missing_target and 'target' in long:
        long = long[long['target'].notna()].reset_index(drop=True)
    return long


def forecast_matrix(rows, pred, index=Y.index, horizon=H):
    """Long rows (article_id, h) + predictions -> articles x horizon array; missing articles -> 0."""
    F = pd.DataFrame({'article_id': rows['article_id'].astype(str).to_numpy(), 'h': rows['h'].to_numpy(),
                      'pred': np.asarray(pred, dtype=float)}).pivot(index='article_id', columns='h', values='pred')
    return F.reindex(index=index, columns=range(1, horizon + 1)).fillna(0).to_numpy(float)


def per_article_mase(cutoff, F, Y=Y, horizon=H):
    """MASE per article (NaN for articles with no movement before the cutoff, which can't be scored)."""
    hist = Y.loc[:, :cutoff]
    act = Y.loc[:, Y.columns > cutoff].iloc[:, :horizon].to_numpy()
    den = hist.diff(axis=1).abs().mean(axis=1).to_numpy()
    err = np.abs(act - np.clip(F, 0, None))
    with np.errstate(divide='ignore', invalid='ignore'):
        return {name: np.where(den > 0, err[:, cols].mean(axis=1) / den, np.nan)
                for name, cols in [('all', slice(0, horizon)), ('w1-w7', slice(0, 7)), ('w8-w13', slice(7, horizon))]}


def mase_score(cutoff, F):
    """Competition score (1 - MASE) x 100, higher is better."""
    return {k: round((1 - np.nanmean(v)) * 100, 2) for k, v in per_article_mase(cutoff, F).items()}


def make_submission(F, index=Y.index):
    F = np.clip(np.asarray(F, dtype=float), 0, None)
    sub = pd.DataFrame(F, index=index, columns=[f'w{i}' for i in range(1, H + 1)]).reset_index()
    assert len(sub) == len(index) and sub.article_id.is_unique and not sub.isna().any().any()
    return sub

## 3. Backtest context (with per-cutoff caching)

In [ ]:
CAT_FEATURES = ['demand_class']
_CONTEXT_CACHE = {}


def clear_context_cache():
    _CONTEXT_CACHE.clear()
    gc.collect()


def make_context(frame, cutoff=None, origin_stride=None, with_train=True, use_cache=True):
    stride_key = tuple(sorted((origin_stride or {}).items()))
    cache_key = (pd.Timestamp(cutoff) if cutoff is not None else None, stride_key, bool(with_train))
    if use_cache and cache_key in _CONTEXT_CACHE:
        return _CONTEXT_CACHE[cache_key]

    train_w, valid_w = split_frame(frame, cutoff)
    cutoff_ts = valid_w['week_start'].max()
    valid = to_long(valid_w, drop_missing_target=False)
    at_cut = valid_w.assign(article_id=valid_w['article_id'].astype(str)).set_index('article_id').reindex(Y.index)
    ctx = {
        'cutoff': cutoff_ts,
        'train': to_long(subsample_origins(train_w, every=origin_stride)) if with_train else None,
        'valid': valid,
        'features': [c for c in valid.columns if c not in LONG_ID_COLS and c != 'target'],
        'Y_hist': Y.loc[:, :cutoff_ts],
        'weeks_since_sale': at_cut['weeks_since_sale'].to_numpy(float),
        'demand_class': at_cut['demand_class'].astype(str).to_numpy(),
    }
    if use_cache:
        _CONTEXT_CACHE[cache_key] = ctx
    return ctx

## 4. Models (Same Expanded Suite, Tuned for Combined `test_shop` + `set_a` + `set_c`)

With `set_a` and `set_c` added to `test_shop`, the global training table is much larger, so tree capacity is scaled back up (`num_leaves=63`, `min_data_in_leaf=500`) while keeping the exact same model family lineup.

In [ ]:
class Fitted:
    """A fitted model: .predict(ctx) -> forecast matrix for ctx's cutoff; .model = underlying object."""
    def __init__(self, predict, model=None):
        self.predict, self.model = predict, model


def _fitted(ctx, predict, model=None):
    return predict(ctx), Fitted(predict, model)


def _repeat(f):
    return np.repeat(np.asarray(f, dtype=float)[:, None], H, axis=1)


# ---------- reference models ----------
def fit_median(ctx, weeks=13):
    return _fitted(ctx, lambda c: _repeat(c['Y_hist'].iloc[:, -weeks:].median(axis=1)))


def fit_naive(ctx):
    return _fitted(ctx, lambda c: _repeat(c['Y_hist'].iloc[:, -1]))


def fit_seasonal_naive(ctx, season=52):
    return _fitted(ctx, lambda c: c['Y_hist'].iloc[:, -season:-season + H].to_numpy(float))


# ---------- LightGBM (L1 & Tweedie) ----------
LGB_PARAMS = dict(objective='l1', learning_rate=0.06, num_leaves=63, min_data_in_leaf=500, max_bin=63,
                  feature_fraction=0.8, bagging_fraction=0.7, bagging_freq=1, lambda_l2=2.0,
                  force_col_wise=True, n_jobs=-1, verbose=-1, seed=SEED)

LGB_TWEEDIE_PARAMS = dict(objective='tweedie', tweedie_variance_power=1.3, learning_rate=0.06,
                          num_leaves=63, min_data_in_leaf=500, max_bin=63,
                          feature_fraction=0.8, bagging_fraction=0.7, bagging_freq=1, lambda_l2=2.0,
                          force_col_wise=True, n_jobs=-1, verbose=-1, seed=SEED + 1)


def _global_predictor(model, feats, transform=None, prep=None):
    """predict(ctx): model output on ctx['valid'] -> (optional transform) -> x scale -> forecast matrix."""
    def predict(c):
        va = c['valid']
        X_va = prep(va[feats]) if prep is not None else va[feats]
        out = model.predict(X_va)
        if transform is not None:
            out = transform(out)
        return forecast_matrix(va, np.clip(out, 0, None) * va['scale'].to_numpy())
    return predict


def fit_lightgbm(ctx, params=LGB_PARAMS, rounds=300, use_weights=True):
    tr, feats = ctx['train'], ctx['features']
    ds = lgb.Dataset(tr[feats], tr['target'], weight=tr['sample_weight'] if use_weights else None,
                     categorical_feature=[c for c in CAT_FEATURES if c in feats])
    model = lgb.train(params, ds, num_boost_round=rounds)
    return _fitted(ctx, _global_predictor(model, feats), model)


def fit_lightgbm_tweedie(ctx, params=LGB_TWEEDIE_PARAMS, rounds=300, use_weights=True):
    tr, feats = ctx['train'], ctx['features']
    ds = lgb.Dataset(tr[feats], tr['target'].clip(lower=0), weight=tr['sample_weight'] if use_weights else None,
                     categorical_feature=[c for c in CAT_FEATURES if c in feats])
    model = lgb.train(params, ds, num_boost_round=rounds)
    return _fitted(ctx, _global_predictor(model, feats), model)


# ---------- CatBoost (MAE / L1) ----------
def fit_catboost(ctx, iterations=550, depth=6, learning_rate=0.07, l2_leaf_reg=5.0, use_weights=True):
    if not HAS_CATBOOST:
        raise ImportError('CatBoost is not installed. Run `%pip install catboost` first.')
    tr, feats = ctx['train'], ctx['features']
    cat_cols = [c for c in CAT_FEATURES if c in feats]

    def prep_cb(df):
        out = df.copy()
        for c in cat_cols:
            out[c] = out[c].astype(str)
        return out

    X_tr = prep_cb(tr[feats])
    model = CatBoostRegressor(
        loss_function='MAE',
        eval_metric='MAE',
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        random_seed=SEED,
        thread_count=-1,
        verbose=0,
    )
    model.fit(X_tr, tr['target'], cat_features=cat_cols,
              sample_weight=tr['sample_weight'] if use_weights else None)
    return _fitted(ctx, _global_predictor(model, feats, prep=prep_cb), model)


# ---------- HistGradientBoostingRegressor (L1) ----------
def fit_hist_gbdt(ctx, max_iter=300, learning_rate=0.06, max_leaf_nodes=63, min_samples_leaf=500, use_weights=True):
    tr, feats = ctx['train'], ctx['features']
    cat_cols = [c for c in CAT_FEATURES if c in feats]
    num_cols = [c for c in feats if c not in cat_cols]
    pre = ColumnTransformer([
        ('num', 'passthrough', num_cols),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ])
    cat_idx = list(range(len(num_cols), len(num_cols) + len(cat_cols)))
    hgb = HistGradientBoostingRegressor(
        loss='absolute_error',
        learning_rate=learning_rate,
        max_iter=max_iter,
        max_leaf_nodes=max_leaf_nodes,
        min_samples_leaf=min_samples_leaf,
        l2_regularization=2.0,
        categorical_features=cat_idx if cat_idx else None,
        random_state=SEED,
    )
    model = make_pipeline(pre, hgb)
    model.fit(tr[feats], tr['target'],
              histgradientboostingregressor__sample_weight=tr['sample_weight'].to_numpy() if use_weights else None)
    return _fitted(ctx, _global_predictor(model, feats), model)


# ---------- shared input pipeline for linear model & MLP ----------
def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))


def make_tabular_preprocessor(features, categorical=(*CAT_FEATURES, 'h')):
    cat = [c for c in categorical if c in features]
    num = [c for c in features if c not in cat]
    return ColumnTransformer([
        ('num', make_pipeline(FunctionTransformer(signed_log),
                              SimpleImputer(strategy='median', add_indicator=True),
                              StandardScaler()), num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
    ])


def _sample(df, n, seed=SEED):
    return df.sample(n=n, random_state=seed) if n and len(df) > n else df


# ---------- linear L1 ----------
def fit_linear_l1(ctx, sample=600_000, alpha=1e-5, epochs=12, target_clip_q=0.999, use_weights=True):
    tr, feats = _sample(ctx['train'], sample), ctx['features']
    model = make_pipeline(make_tabular_preprocessor(feats),
                          SGDRegressor(loss='epsilon_insensitive', epsilon=0.0, alpha=alpha, average=True,
                                       max_iter=epochs, tol=None, random_state=SEED))
    y = tr['target'].clip(upper=tr['target'].quantile(target_clip_q))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model.fit(tr[feats], y, sgdregressor__sample_weight=tr['sample_weight'].to_numpy() if use_weights else None)
    return _fitted(ctx, _global_predictor(model, feats), model)


# ---------- MLP ----------
def fit_mlp(ctx, sample=300_000, hidden=(128, 64), max_iter=25, alpha=1e-4, zero_below=0.5, use_weights=True):
    tr, feats = _sample(ctx['train'], sample), ctx['features']
    model = make_pipeline(make_tabular_preprocessor(feats),
                          MLPRegressor(hidden_layer_sizes=hidden, batch_size=2048, learning_rate_init=1e-3,
                                       alpha=alpha, max_iter=max_iter, early_stopping=True, n_iter_no_change=4,
                                       random_state=SEED))
    y_log = np.log1p(tr['target'].clip(lower=0))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        if use_weights and 'sample_weight' in tr:
            try:
                model.fit(tr[feats], y_log, mlpregressor__sample_weight=tr['sample_weight'].to_numpy())
            except TypeError:
                model.fit(tr[feats], y_log)
        else:
            model.fit(tr[feats], y_log)

    def back(out):
        out = np.expm1(out).clip(0)
        return np.where(out < zero_below, 0, out)
    return _fitted(ctx, _global_predictor(model, feats, back), model)


# ---------- TSB ----------
def tsb_state(Y_hist, alpha=0.1, beta=0.05):
    """Run TSB over the history, vectorised across articles. Returns (p, z) at the last week."""
    X = Y_hist.to_numpy(float)
    p, z = np.zeros(len(X)), np.full(len(X), np.nan)
    for t in range(X.shape[1]):
        d = X[:, t]
        sale = d > 0
        z = np.where(sale, np.where(np.isnan(z), d, z + alpha * (d - z)), z)
        p = p + beta * (sale - p)
    return p, np.nan_to_num(z)


def fit_tsb(ctx, alpha=0.1, beta=0.05, mode='median', threshold=0.5):
    def predict(c):
        p, z = tsb_state(c['Y_hist'], alpha, beta)
        return _repeat(p * z if mode == 'mean' else np.where(p > threshold, z, 0.0))
    return _fitted(ctx, predict)


# ---------- Croston SBA (Syntetos-Boylan Approximation) ----------
def croston_sba_state(Y_hist, alpha=0.1):
    """Vectorized Croston SBA across articles. Returns de-biased rate (1 - alpha/2) * z / p."""
    X = Y_hist.to_numpy(float)
    n = len(X)
    z = np.full(n, np.nan)
    p = np.full(n, np.nan)
    q = np.ones(n)
    for t in range(X.shape[1]):
        d = X[:, t]
        sale = d > 0
        z = np.where(sale, np.where(np.isnan(z), d, z + alpha * (d - z)), z)
        p = np.where(sale, np.where(np.isnan(p), q, p + alpha * (q - p)), p)
        q = np.where(sale, 1.0, q + 1.0)
    rate = np.where((p > 0) & np.isfinite(z), (1.0 - alpha / 2.0) * z / p, 0.0)
    return rate


def fit_croston_sba(ctx, alpha=0.1):
    return _fitted(ctx, lambda c: _repeat(croston_sba_state(c['Y_hist'], alpha)))

## 5. Post-processing (per-model & demand-class aware)

In [ ]:
def zero_no_sales_class(F, ctx, classes=('no_sales',)):
    """Zero out forecasts for articles whose cutoff demand_class is in `classes`."""
    mask = np.isin(ctx['demand_class'], classes)
    return np.where(mask[:, None], 0.0, F)


def dead_to_zero(F, ctx, weeks=8):
    dead = ~(ctx['weeks_since_sale'] < weeks)          # NaN (never sold) counts as dead
    return np.where(dead[:, None], 0.0, F)


def snap_small(F, ctx, below=0.5):
    return np.where(F < below, 0.0, F)


def round_to_int(F, ctx):
    return np.round(F)


def postprocess(F, ctx, steps):
    """Apply `steps` in order; `steps=None` clips >= 0 only."""
    F = np.clip(F, 0, None)
    for step in (steps or ()):
        if step is None:
            continue
        func, kw = (step, {}) if callable(step) else step
        F = func(F, ctx, **kw)
    return F

## 6. Configure Models, Ensembles & Class Router ← edit here

- Keeps the exact same model lineup (`median_13w`, `naive_last`, `seasonal_naive_52`, `lightgbm`, `lightgbm_tweedie`, `catboost`, `hist_gbdt`, `linear_l1`, `tsb`, `croston_sba`, plus `mlp` in full mode) and ensembles (`ens_mean(lgb+cat)`, `ens_median(top_diverse)`, `class_routed`).
- Uses `ORIGIN_STRIDE = {'test_shop': 1, 'set_a': 4, 'set_c': 4}` (or `{'test_shop': 2, 'set_a': 6, 'set_c': 6}` in `FAST_MODE`) so `test_shop` is densely sampled while `set_a` and `set_c` add fast cross-series diversity.

In [ ]:
FAST_MODE = True   # Flip to False for full 4-window stride-1 benchmark + MLP + diagnostics

MODELS = {
    'median_13w':        (fit_median, {'weeks': 13}),
    'naive_last':        (fit_naive, {}),
    'seasonal_naive_52': (fit_seasonal_naive, {}),
    'lightgbm':          (fit_lightgbm, {'rounds': 300}),
    'lightgbm_tweedie':  (fit_lightgbm_tweedie, {'rounds': 300}),
    'hist_gbdt':         (fit_hist_gbdt, {'max_iter': 300}),
    'linear_l1':         (fit_linear_l1, {'sample': 600_000 if FAST_MODE else 1_200_000}),
    'tsb':               (fit_tsb, {'alpha': 0.1, 'beta': 0.05, 'mode': 'median'}),
    'croston_sba':       (fit_croston_sba, {'alpha': 0.1}),
}
if HAS_CATBOOST:
    MODELS['catboost'] = (fit_catboost, {'iterations': 400 if FAST_MODE else 550})
if not FAST_MODE:
    MODELS['mlp'] = (fit_mlp, {'sample': 350_000, 'hidden': (128, 64)})

REFERENCES = ['median_13w', 'naive_last', 'seasonal_naive_52']
BASELINE = 'median_13w'

tree_members = [m for m in ['lightgbm', 'catboost'] if m in MODELS]
diverse_members = [m for m in ['lightgbm', 'lightgbm_tweedie', 'catboost', 'hist_gbdt', 'linear_l1', 'tsb'] if m in MODELS]

ENSEMBLES = {
    'ens_mean(lgb+cat)':       (np.mean, tree_members),
    'ens_median(top_diverse)': (np.median, diverse_members),
}

# Demand-class router: uses TSB on intermittent/lumpy articles and ens_mean(lgb+cat) on smooth/erratic articles
CLASS_ROUTED_NAME = 'class_routed'
CLASS_ROUTE_MAP = {
    'smooth': 'ens_mean(lgb+cat)',
    'erratic': 'ens_mean(lgb+cat)',
    'intermittent': 'tsb',
    'lumpy': 'tsb',
    'no_sales': 'tsb',
}

DEFAULT_POSTPROCESS = [
    (zero_no_sales_class, {'classes': ('no_sales',)}),
    (dead_to_zero, {'weeks': 8}),
    (snap_small, {'below': 0.5}),
]

MODEL_POSTPROCESS = {
    'tsb': [
        (zero_no_sales_class, {'classes': ('no_sales',)}),
        (dead_to_zero, {'weeks': 10}),
    ],
    'croston_sba': [
        (zero_no_sales_class, {'classes': ('no_sales',)}),
        (dead_to_zero, {'weeks': 10}),
        (snap_small, {'below': 0.4}),
    ],
    'lightgbm': DEFAULT_POSTPROCESS,
    'lightgbm_tweedie': DEFAULT_POSTPROCESS,
    'catboost': DEFAULT_POSTPROCESS,
    'hist_gbdt': DEFAULT_POSTPROCESS,
    'linear_l1': DEFAULT_POSTPROCESS,
    'mlp': [
        (zero_no_sales_class, {'classes': ('no_sales',)}),
        (dead_to_zero, {'weeks': 8}),
        (snap_small, {'below': 0.75}),
    ],
}

ENSEMBLE_POSTPROCESS = {
    'ens_mean(lgb+cat)': DEFAULT_POSTPROCESS,
    'ens_median(top_diverse)': DEFAULT_POSTPROCESS,
    CLASS_ROUTED_NAME: [
        (zero_no_sales_class, {'classes': ('no_sales',)}),
        (dead_to_zero, {'weeks': 9}),
        (snap_small, {'below': 0.35}),
    ],
}


def get_post_steps(name):
    if name in ENSEMBLE_POSTPROCESS:
        return ENSEMBLE_POSTPROCESS[name]
    return MODEL_POSTPROCESS.get(name, DEFAULT_POSTPROCESS)


N_WINDOWS = 2 if FAST_MODE else 4
TUNING_WINDOWS = 1
ORIGIN_STRIDE = {'test_shop': 2, 'set_a': 6, 'set_c': 6} if FAST_MODE else {'test_shop': 1, 'set_a': 4, 'set_c': 4}
RUN_DIAGNOSTICS = not FAST_MODE

SITE_SCORES = {'AutoARIMA (site)': -33.7, 'naive (site)': -64.5}
print(f'Mode: {"FAST" if FAST_MODE else "FULL"} | windows={N_WINDOWS} | stride={ORIGIN_STRIDE} | CatBoost={HAS_CATBOOST}')
print('Models:', list(MODELS))

## 7. Rolling backtest (including ensembles & `class_routed`)

In [ ]:
def combine(store, members, how, cutoff):
    return how(np.stack([store[(m, cutoff)] for m in members]), axis=0)


def route_by_demand_class(store, cutoff, ctx, route_map=CLASS_ROUTE_MAP):
    dc = ctx['demand_class']
    fallback = route_map.get('smooth', 'lightgbm')
    out = store[(fallback, cutoff)].copy()
    for cls_name, model_name in route_map.items():
        if (model_name, cutoff) in store:
            mask = (dc == cls_name)
            out[mask] = store[(model_name, cutoff)][mask]
    return out


def _score_rows(name, cutoff, F, F_in, ctx, ctx_in, post_steps, secs):
    rows = []
    for variant, G, c in [('raw', np.clip(F, 0, None), ctx), ('post', postprocess(F, ctx, post_steps), ctx),
                          ('in-sample', postprocess(F_in, ctx_in, post_steps), ctx_in)]:
        rows.append({'model': name, 'cutoff': cutoff.date(), 'variant': variant, 'fit_s': round(secs, 1),
                     **mase_score(c['cutoff'], G)})
    return rows


def backtest(frame, models, ensembles, cutoffs, origin_stride):
    rows, forecasts, forecasts_in, fitted, contexts = [], {}, {}, {}, {}
    for i, cutoff in enumerate(cutoffs):
        t0 = time.perf_counter()
        ctx = make_context(frame, cutoff, origin_stride)
        ctx_in = make_context(frame, cutoff - pd.Timedelta(weeks=H), with_train=False)
        contexts[cutoff] = {k: v for k, v in ctx.items() if k not in ('train', 'valid')}
        src_counts = ctx['train']['source'].value_counts().to_dict() if 'source' in ctx['train'] else {}
        print(f'\n=== cutoff {cutoff.date()} | train {len(ctx["train"]):,} rows {src_counts} | context {time.perf_counter() - t0:.1f}s')
        print(f'  {"model":<24s} {"in-sample":>9s} {"raw":>8s} {"post":>8s}')
        for name, (fit, kw) in models.items():
            t0 = time.perf_counter()
            F, model = fit(ctx, **kw)
            secs = time.perf_counter() - t0
            forecasts[(name, cutoff)], forecasts_in[(name, cutoff)] = F, model.predict(ctx_in)
            if i == 0:
                fitted[name] = model
            new = _score_rows(name, cutoff, F, forecasts_in[(name, cutoff)], ctx, ctx_in, get_post_steps(name), secs)
            rows += new
            print(f'  {name:<24s} {new[2]["all"]:9.2f} {new[0]["all"]:8.2f} {new[1]["all"]:8.2f}   ({secs:.1f}s)')
        for name, (how, members) in ensembles.items():
            if all((m, cutoff) in forecasts for m in members):
                forecasts[(name, cutoff)] = combine(forecasts, members, how, cutoff)
                forecasts_in[(name, cutoff)] = combine(forecasts_in, members, how, cutoff)
                new = _score_rows(name, cutoff, forecasts[(name, cutoff)], forecasts_in[(name, cutoff)],
                                  ctx, ctx_in, get_post_steps(name), 0.0)
                rows += new
                print(f'  {name:<24s} {new[2]["all"]:9.2f} {new[0]["all"]:8.2f} {new[1]["all"]:8.2f}')

        # Demand-class routed forecast
        if all((m, cutoff) in forecasts for m in set(CLASS_ROUTE_MAP.values())):
            forecasts[(CLASS_ROUTED_NAME, cutoff)] = route_by_demand_class(forecasts, cutoff, ctx)
            forecasts_in[(CLASS_ROUTED_NAME, cutoff)] = route_by_demand_class(forecasts_in, cutoff, ctx_in)
            new = _score_rows(CLASS_ROUTED_NAME, cutoff, forecasts[(CLASS_ROUTED_NAME, cutoff)],
                              forecasts_in[(CLASS_ROUTED_NAME, cutoff)], ctx, ctx_in,
                              get_post_steps(CLASS_ROUTED_NAME), 0.0)
            rows += new
            print(f'  {CLASS_ROUTED_NAME:<24s} {new[2]["all"]:9.2f} {new[0]["all"]:8.2f} {new[1]["all"]:8.2f}')

    return pd.DataFrame(rows), forecasts, forecasts_in, fitted, contexts


cutoffs = rolling_cutoffs(frame, n_windows=N_WINDOWS)
tuning_cutoffs, test_cutoffs = cutoffs[:TUNING_WINDOWS], cutoffs[TUNING_WINDOWS:]
print('Tuning window(s):', [c.date() for c in tuning_cutoffs], '| test windows:', [c.date() for c in test_cutoffs])
results, forecasts, forecasts_in, fitted, contexts = backtest(frame, MODELS, ENSEMBLES, cutoffs, ORIGIN_STRIDE)

### Results: tuning window vs test windows

Score = `(1 − MASE) × 100`, **higher is better** (`w8-w13` is used for final model ranking).

In [ ]:
post = results[results.variant == 'post']


def window_mean(df, cuts, cols=('all', 'w1-w7', 'w8-w13')):
    return df[df.cutoff.isin([c.date() for c in cuts])].groupby('model')[list(cols)].mean()


summary = pd.concat({'tuning window': window_mean(post, tuning_cutoffs),
                     'TEST windows (untouched)': window_mean(post, test_cutoffs),
                     'all windows': window_mean(post, cutoffs)}, axis=1).round(2)
summary[('', 'beats baseline on every test window')] = [
    bool((post[(post.model == m) & post.cutoff.isin([c.date() for c in test_cutoffs])].set_index('cutoff')['all']
          > post[(post.model == BASELINE) & post.cutoff.isin([c.date() for c in test_cutoffs])].set_index('cutoff')['all']).all())
    for m in summary.index]
summary = summary.sort_values(('TEST windows (untouched)', 'all'), ascending=False)
display(summary)

print('Per window (all w1-w13):')
display(post.pivot(index='model', columns='cutoff', values='all').loc[summary.index])

## 8. Overfitting check

In [ ]:
ins = results[results.variant == 'in-sample'].groupby('model')['all'].mean()
oos = post.groupby('model')['all'].mean()
gap = pd.DataFrame({'in-sample (trained-on weeks)': ins, 'out-of-sample (unseen weeks)': oos})
gap['gap'] = gap.iloc[:, 0] - gap.iloc[:, 1]
gap['excess gap vs baseline'] = gap['gap'] - gap.loc[BASELINE, 'gap']
display(gap.loc[summary.index].round(2))

**LightGBM learning curve** (runs when `RUN_DIAGNOSTICS = True`).

In [ ]:
LC_ROUNDS = [25, 50, 100, 150, 200, 300, 400, 600]


def lgb_learning_curve(frame, cutoff, rounds_grid=LC_ROUNDS, params=LGB_PARAMS):
    ctx = make_context(frame, cutoff, ORIGIN_STRIDE)
    ctx_in = make_context(frame, cutoff - pd.Timedelta(weeks=H), with_train=False)
    feats, tr = ctx['features'], ctx['train']
    ds = lgb.Dataset(tr[feats], tr['target'], weight=tr['sample_weight'], categorical_feature=CAT_FEATURES)
    model = lgb.train(params, ds, num_boost_round=max(rounds_grid))
    lgb_steps = get_post_steps('lightgbm')
    rows = []
    for r in rounds_grid:
        row = {'rounds': r}
        for label, c in [('trained-on weeks', ctx_in), ('unseen weeks', ctx)]:
            va = c['valid']
            F = forecast_matrix(va, np.clip(model.predict(va[feats], num_iteration=r), 0, None) * va['scale'].to_numpy())
            row[label] = mase_score(c['cutoff'], postprocess(F, c, lgb_steps))['all']
        rows.append(row)
    return pd.DataFrame(rows).set_index('rounds')


if RUN_DIAGNOSTICS and 'lightgbm' in MODELS:
    t0 = time.perf_counter()
    lc = lgb_learning_curve(frame, tuning_cutoffs[0])
    print(f'learning curve ({time.perf_counter() - t0:.1f}s)')
    display(lc)
    ORANGE = '#eb6834'
    fig, ax = plt.subplots(figsize=(9, 4))
    for col, color in [('trained-on weeks', ORANGE), ('unseen weeks', BLUE)]:
        ax.plot(lc.index, lc[col], color=color, linewidth=2, marker='o', markersize=5, label=col)
        ax.annotate(col, (lc.index[-1], lc[col].iloc[-1]), textcoords='offset points', xytext=(6, 0),
                    va='center', color=INK, fontsize=9)
    ax.axvline(MODELS['lightgbm'][1].get('rounds', 300), color=INK, linewidth=1, linestyle='--')
    ax.set_xlabel('boosting rounds'); ax.set_ylabel('score (higher is better)')
    ax.set_xlim(0, lc.index[-1] * 1.25)
    ax.legend(frameon=False, loc='lower right')
    ax.set_title(f'LightGBM score vs boosting rounds (window {tuning_cutoffs[0].date()})')
    plt.tight_layout(); plt.show()
else:
    print('Skipped LightGBM learning curve (FAST_MODE=True). Set RUN_DIAGNOSTICS = True to run.')

## 9. How good is it, and does it beat the site's baselines?

In [ ]:
cal = post.pivot(index='model', columns='cutoff', values='w1-w7').loc[summary.index]
print('Live-leaderboard weeks (w1-w7) per backtest window:')
display(cal.round(2))
nearest = (cal.loc['naive_last'] - SITE_SCORES['naive (site)']).abs().idxmin()
best = summary.index[~summary.index.isin(REFERENCES)][0]
print(f"Site naive on the real hidden w1-w7: {SITE_SCORES['naive (site)']} | closest backtest window: {nearest} "
      f"(our naive {cal.loc['naive_last', nearest]:.1f})")
print(f"In that window: {best} {cal.loc[best, nearest]:.1f}, median baseline {cal.loc[BASELINE, nearest]:.1f}, "
      f"vs site AutoARIMA {SITE_SCORES['AutoARIMA (site)']} on the real weeks.")

## 10. Should forecasts be rounded to whole units?

In [ ]:
rnd = []
for (m, c), F in forecasts.items():
    G = postprocess(F, contexts[c], get_post_steps(m))
    rnd.append({'model': m, 'cutoff': c.date(), 'decimals': mase_score(c, G)['all'],
                'rounded': mase_score(c, np.round(G))['all']})
rnd = pd.DataFrame(rnd)
rnd_t = rnd.groupby('model')[['decimals', 'rounded']].mean().round(2)
rnd_t['rounding changes score by'] = (rnd_t['rounded'] - rnd_t['decimals']).round(2)
rnd_t['rounding better on # windows'] = rnd.assign(b=rnd.rounded > rnd.decimals).groupby('model')['b'].sum()
display(rnd_t.loc[summary.index])

## 11. Where the error comes from (by demand class)

In [ ]:
c0 = tuning_cutoffs[0]
ctx0 = contexts[c0]
by_class = {}
for m in summary.index:
    G = postprocess(forecasts[(m, c0)], ctx0, get_post_steps(m))
    by_class[m] = pd.Series(per_article_mase(c0, G)['all']).groupby(ctx0['demand_class']).mean()
cls = pd.DataFrame(by_class).T.round(3)
cls.loc['articles'] = pd.Series(ctx0['demand_class']).value_counts()
display(cls[[c for c in ['smooth', 'erratic', 'intermittent', 'lumpy', 'no_sales'] if c in cls]])

### TSB parameter sensitivity (runs when `RUN_DIAGNOSTICS = True`)

In [ ]:
if RUN_DIAGNOSTICS:
    tsb_steps = get_post_steps('tsb')
    grid = []
    for a in [0.05, 0.1, 0.2, 0.3]:
        for b in [0.02, 0.05, 0.1]:
            for thr in [0.4, 0.5, 0.6]:
                sc = {c: mase_score(c, postprocess(fit_tsb({'Y_hist': Y.loc[:, :c]}, a, b, 'median', thr)[0],
                                                   contexts[c], tsb_steps))['all'] for c in cutoffs}
                grid.append({'alpha': a, 'beta': b, 'threshold': thr,
                             'tuning window': round(np.mean([sc[c] for c in tuning_cutoffs]), 2),
                             'test windows (info only)': round(np.mean([sc[c] for c in test_cutoffs]), 2) if test_cutoffs else np.nan})
    display(pd.DataFrame(grid).sort_values('tuning window', ascending=False).head(10))
else:
    print('Skipped TSB grid scan (FAST_MODE=True). Set RUN_DIAGNOSTICS = True to run.')

## 12. LightGBM feature importance

In [ ]:
if 'lightgbm' in fitted:
    m = fitted['lightgbm'].model
    imp = pd.Series(m.feature_importance('gain'), index=m.feature_name()).sort_values()
    imp = (imp / imp.sum() * 100).tail(20)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(imp.index, imp.values, color=BLUE, height=0.6)
    ax.grid(axis='y', visible=False)
    ax.set_xlabel('% of total gain')
    ax.set_title('LightGBM: top 20 features by gain (most recent window)')
    plt.tight_layout(); plt.show()

## 13. Final fit and submission

Automatically picks the top model/ensemble/router on the tuning window (`w8-w13`), refits all required base models on **all** `test_shop` + `set_a` + `set_c` history, applies `get_post_steps(FINAL_MODEL)`, writes `submission_<model>.csv`, and triggers a Colab download.

In [ ]:
clear_context_cache()   # Free backtest train frames before final full-history fit

candidates = [m for m in summary.index if m not in REFERENCES]
FINAL_MODEL = window_mean(post, tuning_cutoffs).loc[candidates, 'w8-w13'].idxmax()
SAVE = True
AUTO_DOWNLOAD_COLAB = True

def resolve_required_base_models(name):
    if name == CLASS_ROUTED_NAME:
        needed = set()
        for target in CLASS_ROUTE_MAP.values():
            needed.update(resolve_required_base_models(target))
        return sorted(needed)
    if name in ENSEMBLES:
        return list(ENSEMBLES[name][1])
    return [name]


members = resolve_required_base_models(FINAL_MODEL)
print(f'Final model: {FINAL_MODEL} | base models: {members}')
print(f'Tuning window w8-w13: {summary.loc[FINAL_MODEL, ("tuning window", "w8-w13")]} '
      f'| TEST windows all: {summary.loc[FINAL_MODEL, ("TEST windows (untouched)", "all")]} '
      f'(baseline {summary.loc[BASELINE, ("TEST windows (untouched)", "all")]})')

ctx_final = make_context(frame, cutoff=None, origin_stride=ORIGIN_STRIDE)
print(f'Training rows: {len(ctx_final["train"]):,} | forecast origin: {ctx_final["cutoff"].date()}')
final_fc = {}
for m in members:
    t0 = time.perf_counter()
    fit, kw = MODELS[m]
    final_fc[m] = fit(ctx_final, **kw)[0]
    print(f'  fitted {m} ({time.perf_counter() - t0:.1f}s)')

def build_final_forecast(name):
    if name == CLASS_ROUTED_NAME:
        dc = ctx_final['demand_class']
        fallback_name = CLASS_ROUTE_MAP.get('smooth', 'lightgbm')
        out = build_final_forecast(fallback_name).copy()
        for cls_name, model_name in CLASS_ROUTE_MAP.items():
            mask = (dc == cls_name)
            out[mask] = build_final_forecast(model_name)[mask]
        return out
    if name in ENSEMBLES:
        how, ens_members = ENSEMBLES[name]
        return how(np.stack([final_fc[m] for m in ens_members]), axis=0)
    return final_fc[name]

F_final = postprocess(build_final_forecast(FINAL_MODEL), ctx_final, get_post_steps(FINAL_MODEL))
sub = make_submission(F_final)

print(f'Articles forecast 0 for all 13 weeks: {(F_final.sum(axis=1) == 0).sum():,} of {len(sub):,}')
print(f'Mean forecast per article-week: {F_final.mean():.2f} (last 13 weeks of history: {Y.iloc[:, -13:].to_numpy().mean():.2f})')
display(sub.head())

if SAVE:
    SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
    path = SUBMISSION_DIR / f'submission_{re.sub(r"[^A-Za-z0-9]+", "_", FINAL_MODEL).strip("_")}.csv'
    sub.to_csv(path, index=False)
    print('Saved', path)
    if IN_COLAB and AUTO_DOWNLOAD_COLAB:
        try:
            from google.colab import files
            files.download(str(path))
        except Exception as e:
            print('Auto-download skipped:', e)

## 14. All model scores

In [ ]:
test_scores = summary[('TEST windows (untouched)', 'all')].rename(index=lambda m: f'{m} (final)' if m == FINAL_MODEL else m)
bars = pd.concat([test_scores, pd.Series(SITE_SCORES)]).sort_values()
kind = ['site' if m in SITE_SCORES else 'reference' if m in REFERENCES else 'ours' for m in bars.index]

fig, ax = plt.subplots(figsize=(10, 0.42 * len(bars) + 1.4))
for y, (m, v, k) in enumerate(zip(bars.index, bars.values, kind)):
    style = {'ours': dict(color=BLUE), 'reference': dict(color=GREY),
             'site': dict(facecolor='white', edgecolor=INK, linewidth=1.2)}[k]
    ax.barh(y, v, height=0.62, **style)
    ax.text(v - 0.8, y, f'{v:.1f}', va='center', ha='right', color=INK, fontsize=9)
ax.set_yticks(range(len(bars)), bars.index)
ax.axvline(0, color=INK, linewidth=0.8)
ax.set_xlim(bars.min() * 1.12, max(bars.max(), 0) + 2)
ax.grid(axis='y', visible=False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=BLUE, label='our models'), Patch(color=GREY, label='reference forecasts (same backtest)'),
                   Patch(facecolor='white', edgecolor=INK, label='site scores (real hidden weeks)')],
          frameon=False, loc='upper left')
ax.set_xlabel('score = (1 − MASE) × 100, higher is better')
ax.set_title('Model scores: mean of the test windows')
plt.tight_layout(); plt.show()